# Figure 4: Receptive Fields across Multiple Depths

This notebook characterizes receptive fields (RFs) and spatial retinotopy measured under multi-depth visual stimulation (`SpheresPermTubeReward_multidepth`), providing the full set of single-session and population figures:

### Panels:
- **A**: Model formulation schematic ($\Delta F/F_0 = S\mathbf{b}$)
- **B**: Reconstructed visual stimulus frames across 8 depths (5 to 640 cm) under multi-depth stimulation
- **C**: Depth tuning curves and multi-depth RF maps for 3 example V1 neurons (ROIs 53, 118, 623 of `PZAH17.1e_S20250318`)
- **D**: 3D receptive field volumes rendered as nested isosurfaces (50-98% of peak, colour and opacity ramping to the core), on a clean white background with faint floor/wall marginal projections
- **E**: Population correlation: Preferred receptive field depth vs Preferred trial-average depth
- **F, G, H**: Spatial retinotopic organization of an example V1 FOV (`PZAG16.3c_S20250317`):
  - **F**: Preferred Azimuth (degrees)
  - **G**: Preferred Elevation (degrees)
  - **H**: Preferred Virtual Depth (cm) with anatomical 2P mean image inset
- **I, J, K**: Mean pairwise |Δ RF azimuth|, |Δ RF elevation| and |Δ log₂ preferred depth| between depth-tuned neurons vs distance between cells, across all V1 sessions of `hey2_3d-vision_foodres_20220101` (hierarchical mouse/session bootstrap 95% CI)
- **Full Figure**: Unified publication canvas assembling Panels A–K

In [ ]:
%reload_ext autoreload
%autoreload 2

import pickle
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

import flexiznam as flz
from cottage_analysis.analysis import spheres
from cottage_analysis.analysis.spheres import rf_analysis
from cottage_analysis.plotting import rf_plots, depth_selectivity_plots, rsof_plots, plotting_utils, style
from cottage_analysis.pipelines import pipeline_utils
from v1_depth_map.figure_utils import receptive_fields as rf_utils
from v1_depth_map.figure_utils import rf_distance
from v1_depth_map.revisions.revision_sessions import sessions
from v1_depth_map.paths import get_figures_roots

In [ ]:
# Register the manuscript font faces (Arial regular + bold + italic, Arial Narrow) and
# apply the publication rcParams: vector fonttypes, font sizes, tick/label padding.
# `style.savefig` then expands the SVG `font:` shorthand so Illustrator reads the
# family, size and weight correctly - see cottage_analysis.plotting.style for both.
from cottage_analysis.plotting import style
from cottage_analysis.plotting.style import CM, FONTSIZE_DICT

style.setup_figure_fonts()

In [ ]:
import os

# RF fits used by every RF panel: "neurons_df_spks.pickle" (RFs refitted on deconvolved
# spikes, used in the manuscript; other columns stay dF/F) or "neurons_df.pickle" (dF/F).
# Outputs and caches get a suffix.
RF_FILE = "neurons_df_spks.pickle"
OUT_SFX = "" if RF_FILE == "neurons_df.pickle" else "_" + RF_FILE[len("neurons_df_"):-len(".pickle")]
print(f"RF fits from {RF_FILE}, output suffix {OUT_SFX!r}")

project = "colasa_3d-vision_revisions"
flexilims_session = flz.get_flexilims_session(project)
flexilims_session_hey2 = flz.get_flexilims_session("hey2_3d-vision_foodres_20220101")
READ_ROOT, SAVE_ROOT = get_figures_roots(flexilims_session)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

## Load Population Data


In [ ]:
# Load all multi-depth sessions for population correlation
multi_depth_sessions = [k for k, v in sessions.items() if v == "multidepth"]

all_sig_m, all_sig_ipsi_m, neurons_df = rf_analysis.load_sig_rf(
    flexilims_session=flexilims_session,
    session_list=multi_depth_sessions,
    n_std=6,
    filter_datasets={"annotated": True},
    use_multidepth=True,
    use_cols=None,
    sphere_presentation_mask=None,
    rf_file=RF_FILE,
)

select_neurons = (neurons_df["iscell"] == 1) & (
    depth_selectivity_plots.common_utils.one_sided_pval_from_spearman(
        neurons_df["depth_tuning_test_spearmanr_rval_closedloop"],
        neurons_df["depth_tuning_test_spearmanr_pval_closedloop"],
    )
    < 0.05
)
neurons_df["is_depth_selective"] = select_neurons
depth_neurons = neurons_df[select_neurons & (neurons_df["rf_sig"] == 1)].copy()
print(
    f"Loaded {len(depth_neurons)} depth-selective neurons with significant RFs across {len(multi_depth_sessions)} sessions."
)

## Load Example Session Data


In [ ]:
# Load example session data for stimulus frames, tuning curves, and receptive fields
example_session = "PZAH17.1e_S20250318"

frames_rf, imaging_df_rf = spheres.regenerate_frames_all_recordings(
    session_name=example_session,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward_multidepth",
    photodiode_protocol=5,
    return_volumes=True,
    resolution=1,
    verbose=False,
    is_multidepth=True,
)
plt.close("all")

neurons_df_example = pipeline_utils.load_neurons_df(
    pipeline_utils.create_neurons_ds(
        session_name=example_session,
        flexilims_session=flexilims_session,
        project=None,
        conflicts="skip",
    ),
    rf_file=RF_FILE,
)

_, trials_df_example = spheres.sync_all_recordings(
    session_name=example_session,
    flexilims_session=flexilims_session,
    filter_datasets={"anatomical_only": 3, "annotated": True},
    recording_type="two_photon",
    protocol_base="SpheresPermTubeReward",
    photodiode_protocol=5,
    return_volumes=True,
)

# Load suite2p ROI metadata for FOV spatial plots
suite2p_ds = flz.get_datasets(
    flexilims_session=flexilims_session,
    origin_name=example_session,
    dataset_type="suite2p_rois",
    filter_datasets={"anatomical_only": 3, "annotated": True},
    allow_multiple=False,
    return_dataseries=False,
)
iscell = np.load(suite2p_ds.path_full / "plane0" / "iscell.npy", allow_pickle=True)[
    :, 0
]
neurons_df_example["iscell"] = iscell
stat = np.load(suite2p_ds.path_full / "plane0" / "stat.npy", allow_pickle=True)
ops = np.load(suite2p_ds.path_full / "plane0" / "ops.npy", allow_pickle=True).item()

In [ ]:
# Load a second session for the FOV retinotopy panels (F, G, H): the example
# neurons come from `example_session`, but this FOV has the cleaner gradient
fov_session =  "PZAG16.3c_S20250317" # "PZAH8.2h_S20230302" #
is_revision_data = True

neurons_df_fov = pipeline_utils.load_neurons_df(
    pipeline_utils.create_neurons_ds(
        session_name=fov_session,
        flexilims_session=flexilims_session if is_revision_data else flexilims_session_hey2,
        project=None,
        conflicts="skip",
    ),
    rf_file=RF_FILE,
)

filt = {"anatomical_only": 3}
if is_revision_data:
    filt["annotated"] = True
suite2p_ds_fov = flz.get_datasets(
    flexilims_session=flexilims_session if is_revision_data else flexilims_session_hey2,
    origin_name=fov_session,
    dataset_type="suite2p_rois",
    filter_datasets=filt,
    allow_multiple=False,
    return_dataseries=False,
)
neurons_df_fov["iscell"] = np.load(
    suite2p_ds_fov.path_full / "plane0" / "iscell.npy", allow_pickle=True
)[:, 0]
stat_fov = np.load(suite2p_ds_fov.path_full / "plane0" / "stat.npy", allow_pickle=True)
ops_fov = np.load(
    suite2p_ds_fov.path_full / "plane0" / "ops.npy", allow_pickle=True
).item()

## Load Pairwise RF Distance Data

In [ ]:
# Pairwise RF / depth distance for panels I, J, K. These come from the original
# single-depth V1 sessions, not the multi-depth revision sessions used above, so
# they use their own flexilims project and the cache in its figures root.
v1_project = "hey2_3d-vision_foodres_20220101"
flexilims_session_v1 = flz.get_flexilims_session(v1_project)
_, SAVE_ROOT_V1 = get_figures_roots(flexilims_session_v1)

recompute_rf_distance = False  # True: rebuild the pairwise-distance dataframe and re-cache; False: load the cached dataframe

pairwise_cache = SAVE_ROOT_V1 / "rf_supp" / f"pairwise_distance_all_sessions{OUT_SFX}.pkl"
pairwise_df = rf_distance.load_pairwise_distances(
    flexilims_session_v1,
    pairwise_cache,
    recompute=recompute_rf_distance or not pairwise_cache.exists(),
    rf_file=RF_FILE,
)
rf_dist_centers, rf_dist_means, rf_dist_ci_low, rf_dist_ci_high, rf_dist_counts = (
    rf_distance.hierarchical_binned_means(pairwise_df)
)
# Expected curve if cells were shuffled across positions within each session
rf_dist_null = rf_distance.shuffle_null(pairwise_df)
print(
    f"{rf_dist_counts['n_sessions']} sessions from {rf_dist_counts['n_mice']} mice, "
    f"{rf_dist_counts['n_pairs']:,} pairs; "
    f"median pairs/bin = {rf_dist_counts['median_pairs_per_bin']:,}"
)
for icol, col in enumerate(rf_distance.CI_COLS):
    width = np.nanmedian(rf_dist_ci_high[icol] - rf_dist_ci_low[icol])
    print(
        f"{col:>28}: median CI width {width:.4f}, "
        f"shuffle null {np.nanmin(rf_dist_null[:, icol]):.3f}-{np.nanmax(rf_dist_null[:, icol]):.3f}"
    )

## Shared Settings & Panel Helpers

Every panel is drawn by a function that takes the target figure (or axes) and the
axes rectangles, so the same code produces the standalone panels below and the
assembled figure at the end of the notebook. Styling knobs (example neurons,
colormaps, isosurface levels, marginal opacity) live at the top of the cell.


In [ ]:
# ------------------------------------------------------------------ constants
# Example neurons: near / middle / far depth-tuned, with distinct azimuth and
# elevation tuning (session PZAH17.1e_S20250318)
SELECT_ROIS = [53, 118, 623]
# One sequential colormap per example neuron, shared by the RF maps and 3D volumes
EXAMPLE_CMAPS = ["Purples", "Greens", "Oranges"]
EXAMPLE_COLORS = [plt.get_cmap(name)(0.85) for name in EXAMPLE_CMAPS]

FOV_WIDTH = 661


# -------------------------------------------------------------- layout helpers
# Centimetre-based layout helpers, shared so panel letters stay at
# FONTSIZE_DICT["panel"] across every figure.
from cottage_analysis.plotting.style import rect_cm, panel_letter

# NOTE: when passing a rect to `plot_rf` or `plot_stimulus_frame`, `y` is the
# *top* of the stack of per-depth axes and `h` its total height, since those
# helpers grow downwards.


# ------------------------------------- stimulus, example neurons, population, FOV
def plot_stimulus_frames(
    fig, position, frames_contra, start_frame, depths, fontsize_dict=None
):
    """Reconstructed stimulus frame at each of the 8 depths."""
    plt.figure(fig.number)
    rf_plots.plot_stimulus_frame(
        frame=frames_contra[:, start_frame],
        idepth=None,
        depths=depths,
        position=position,
        fontsize_dict=fontsize_dict or FONTSIZE_DICT,
        plot_prop=0.9,
    )


def plot_example_neurons(
    fig,
    neurons_df,
    trials_df,
    tuning_rect,
    rf_rect,
    dx,
    rois=None,
    colors=None,
    ndepths=8,
    fontsize_dict=None,
):
    """Depth tuning curve + multi-depth RF map per example neuron.

    `tuning_rect` and `rf_rect` are the rects of the leftmost neuron; each
    subsequent neuron is shifted right by `dx` (figure fractions throughout).
    """
    fig = plt.figure(fig.number)
    rois = SELECT_ROIS if rois is None else rois
    colors = EXAMPLE_COLORS if colors is None else colors
    fontsize_dict = fontsize_dict or FONTSIZE_DICT
    axes = []
    for iroi, (roi, linecolor) in enumerate(zip(rois, colors)):
        ax = fig.add_axes([tuning_rect[0] + dx * iroi, *tuning_rect[1:]])
        depth_selectivity_plots.plot_depth_tuning_curve(
            neurons_df=neurons_df,
            trials_df=trials_df,
            roi=roi,
            linecolor=linecolor,
            rs_thr=None,
            plot_fit=True,
            plot_smooth=False,
            linewidth=1,
            closed_loop=1,
            fontsize_dict=fontsize_dict,
            markersize=5,
            markeredgecolor="w",
            ylim_precision_base=5,
            ylim_precision=2,
        )
        ax.set_ylabel("")
        if iroi == len(rois) // 2:
            ax.set_xlabel("Virtual depth (cm)", fontsize=fontsize_dict["label"])
        else:
            ax.set_xlabel("")
        # The tuning curve ticks every depth; keep every other one to declutter
        xticks = ax.get_xticks()
        xlabels = [t.get_text() for t in ax.get_xticklabels()]
        ax.set_xticks(
            xticks[::2], xlabels[::2], rotation=45, fontsize=fontsize_dict["tick"]
        )
        ax.tick_params(length=1.5)
        axes.append(ax)
        rf_plots.plot_rf(
            neurons_df=neurons_df,
            roi=roi,
            ndepths=ndepths,
            frame_shape=(16, 24),
            position=[rf_rect[0] + dx * iroi, *rf_rect[1:]],
            plot_prop=0.9,
            xlabel="Azimuth (degrees)" if iroi == len(rois) // 2 else "",
            ylabel="Elevation (degrees)" if iroi == 0 else "",
            fontsize_dict=fontsize_dict,
            use_multidepth=True,
            plot_yticklabels=iroi == 0,
        )

    return axes


def plot_rf_vs_tuning_depth(fig, depth_neurons, rect, fontsize_dict=None):
    """Population: RF preferred depth vs trial-averaged preferred depth."""
    xcol = "preferred_depth_closedloop"
    ycol = "rf_preferred_depth_closedloop_multidepth"
    # Depths are stored in m; plot in cm
    depth_cm = depth_neurons[[xcol, ycol]] * 100
    rsof_plots.plot_2d_hist(
        neurons_df=depth_cm,
        ycol="rf_preferred_depth_closedloop_multidepth",
        xcol="preferred_depth_closedloop",
        fig=fig,
        position=rect,
        ylabel="Preferred receptive field depth (cm)",
        xlabel="Preferred trial-average depth (cm)",
        plot_diagonal=True,
        aspect_equal=True,
        fontsize_dict=fontsize_dict or FONTSIZE_DICT,
        color_by_density=False,
        plot_contours=True,
        scatter_kwargs={"s": 5, "alpha": 0.5, "color": "k"},
        contour_kwargs={"color": "gray"},
    )


def plot_fov_retinotopy(
    fig,
    rects,
    neurons_df_fov,
    stat_fov,
    ops_fov,
    depths,
    inset_rect=None,
    fov_width=FOV_WIDTH,
    fontsize_dict=None,
):
    """Preferred azimuth / elevation / depth across the example FOV.

    `rects` is the three axes rects; `inset_rect` places the anatomical meanImg
    inset inside the depth map (skipped when None).
    """
    plt.figure(fig.number)
    fontsize_dict = fontsize_dict or FONTSIZE_DICT
    maps = [
        ("rf_azi", cm.YlOrRd.reversed()),
        ("rf_ele", cm.YlOrRd.reversed()),
        ("preferred_depth_closedloop", cm.cool.reversed()),
    ]
    images = []
    for rect, (col, cmap) in zip(rects, maps):
        fig.add_axes(rect)
        images.append(
            depth_selectivity_plots.plot_example_fov(
                neurons_df=neurons_df_fov,
                ops=ops_fov,
                stat=stat_fov,
                ndepths=len(depths),
                col=col,
                cmap=cmap,
                background_color=np.array([0, 0, 0]),
                fontsize_dict=fontsize_dict,
                fov_width=fov_width,
            )
        )
    if inset_rect is not None:
        plotting_utils.plot_white_rectangle(
            inset_rect[0] - 0.005,
            inset_rect[1] - 0.005,
            inset_rect[2] + 0.01,
            inset_rect[3] + 0.01,
        )
        fig.add_axes(inset_rect)
        depth_selectivity_plots.plot_fov_mean_img(
            ops_fov["meanImg"], fov_width=fov_width, vmax=4000
        )
    return images


# ------------------------------------------------------------- 3D RF volumes
# Drawing helpers live in v1_depth_map.figure_utils.receptive_fields (rf_utils)
def rf_volume(neurons_df, roi, ndepths, azi_slice, frame_shape=(16, 24)):
    """Fold-averaged, rectified multi-depth RF volume, cropped in azimuth."""
    coef = neurons_df.loc[roi, "rf_coef_closedloop_multidepth"][:, :-1]
    coef = coef.reshape(coef.shape[0], ndepths, *frame_shape)[..., azi_slice]
    return np.maximum(np.mean(coef, axis=0), 0)


def plot_rf_volumes_3d(
    ax,
    neurons_df,
    depths_arr,
    rois=None,
    cmaps=None,
    colors=None,
    fontsize_dict=None,
    elev=28,
    azim=-50,
    tickpad=-2,
    labelpad=-10,
):
    """3D receptive-field volumes of the example neurons in visual space.

    The drawing (isosurface stack, peak marker, marginal projections, cube
    styling) and the displayed window live in
    `v1_depth_map.figure_utils.receptive_fields`.

    `tickpad` (cube to tick labels) and `labelpad` (tick labels to axis labels)
    are forwarded to `rf_utils.style_rf_3d_axes`; both are negative because
    mplot3d spaces them generously for a panel this small.
    """
    rois = SELECT_ROIS if rois is None else rois
    cmaps = EXAMPLE_CMAPS if cmaps is None else cmaps
    colors = EXAMPLE_COLORS if colors is None else colors
    azi_slice = slice(
        int(rf_utils.AZI_LIM[0] / rf_utils.AZI_RESOLUTION),
        int(rf_utils.AZI_LIM[1] / rf_utils.AZI_RESOLUTION),
    )
    for roi, cmap_name, color in zip(rois, cmaps, colors):
        volume = rf_volume(neurons_df, roi, len(depths_arr), azi_slice)
        rf_utils.add_rf_isosurfaces(ax, volume, cmap_name)
        peak = rf_utils.add_rf_peak_marker(ax, volume, color)
        rf_utils.add_rf_marginals(ax, volume, color, peak)
    rf_utils.style_rf_3d_axes(
        ax,
        depths_arr,
        fontsize_dict,
        elev=elev,
        azim=azim,
        tickpad=tickpad,
        labelpad=labelpad,
    )
    return ax

## Panels B & C: Multi-Depth Stimuli & Example Neuron RFs


In [ ]:
depths = sorted(neurons_df.best_depth.unique())
ndepths = len(depths)

# A frame where spheres are present at every depth
frames_contra = frames_rf[..., frames_rf.shape[-1] // 2 :]
has_data = np.any(frames_contra, axis=(2, 3))
start_frame = np.where(np.sum(has_data, axis=0) == ndepths)[0][2450]

fig = plt.figure(figsize=(18 / 2.54, 18 / 2.54))
plot_stimulus_frames(fig, [0, 0.74, 0.3, 0.4], frames_contra, start_frame, depths)
axes = plot_example_neurons(
    fig,
    neurons_df=neurons_df_example,
    trials_df=trials_df_example,
    tuning_rect=[0.267, 0.9, 0.07, 0.05],
    rf_rect=[0.055, 0.8, 0.5, 0.5],
    dx=0.13,
    ndepths=ndepths,
)
plot_rf_vs_tuning_depth(fig, depth_neurons, [0.68, 0.74, 0.25, 0.21])

style.savefig(
    SAVE_ROOT / f"fig_receptive_fields_examples{OUT_SFX}.svg", bbox_inches="tight", fig=fig
)
plt.show()

## Exploratory: Plotly Volume Renderer (not used in the assembled figure)


In [ ]:
# Plot 3D spatio-temporal receptive fields in visual space (Plotly volume)
if False:
    depths = sorted(neurons_df.best_depth.unique())
    rf_plots.plot_rf_3d(
        neurons_df_example,
        SELECT_ROIS,
        depths,
        SAVE_ROOT / f"fig_receptive_fields_3d_rfs{OUT_SFX}.pdf",
        dict(label=10, tick=10),
    )

## Panel D: 3D Receptive Fields (Nested Isosurfaces)

This polished version matches the original viewing perspective (Azimuth on the left-front, Depth on the right-front, Elevation vertical) while using 12 nested isosurfaces spanning 50-98% of peak, with colour and opacity ramping from a pale envelope to a dark core on a clean white background with publication lighting, drop lines, and 2D marginal projections on the floor (Azimuth $\times$ Depth) and walls.


In [ ]:
depths_arr = np.array(sorted(neurons_df.best_depth.unique()))

fig = plt.figure(figsize=(9 / 2.54, 8 / 2.54), dpi=300)
fig.patch.set_facecolor("white")
plot_rf_volumes_3d(
    fig.add_subplot(111, projection="3d"),
    neurons_df_example,
    depths_arr,
    elev=38,
    azim=-40,
    tickpad=-2,
    labelpad=-5,
)

for suffix in ("svg", "pdf"):
    style.savefig(
        SAVE_ROOT / f"fig_receptive_fields_3d_rfs_polished{OUT_SFX}.{suffix}",
        bbox_inches="tight",
        fig=fig,
        dpi=600,  # resolution of the rasterised volumes
    )
plt.show()

## Panels F, G, H: Spatial Retinotopy in V1 FOV

In [ ]:
depths = sorted(neurons_df.best_depth.unique())

fig = plt.figure(figsize=(18 / 2.54, 8 / 2.54), dpi=300)
plot_fov_retinotopy(
    fig,
    rects=[
        [0.04, 0.05, 0.28, 0.88],
        [0.37, 0.05, 0.28, 0.88],
        [0.70, 0.05, 0.28, 0.88],
    ],
    neurons_df_fov=neurons_df_fov,
    stat_fov=stat_fov,
    ops_fov=ops_fov,
    depths=depths,
    inset_rect=[0.86, 0.66, 0.08, 0.08],
)

style.savefig(SAVE_ROOT / f"fig_receptive_fields_fov{OUT_SFX}.svg", bbox_inches="tight", fig=fig)
plt.show()
# Gradient of each map across the FOV, on the neurons plot_example_fov colours
# (significant RF and iscell; plot_example_fov added rf_azi/rf_ele/center_x/center_y
# to neurons_df_fov). Shuffling treats neighbouring cells as independent, so the
# p-values are optimistic. Directions are in image coordinates (x right, y down),
# before the display flip of the x axis.
from cottage_analysis.analysis import roi_location
from cottage_analysis.analysis.spheres import rf_fitting

coef = np.stack(neurons_df_fov["rf_coef_closedloop"].values)
coef_ipsi = np.stack(neurons_df_fov["rf_coef_ipsi_closedloop"].values)
rf_sig, _ = rf_fitting.find_sig_rfs(
    np.swapaxes(np.swapaxes(coef, 0, 2), 0, 1),
    np.swapaxes(np.swapaxes(coef_ipsi, 0, 2), 0, 1),
    n_std=6,
)
fov_cells = neurons_df_fov[(rf_sig == 1) & (neurons_df_fov["iscell"] == 1)]
um_per_px = FOV_WIDTH / ops_fov["Ly"]

for label, col, unit, transform in [
    ("RF azimuth", "rf_azi", "deg", None),
    ("RF elevation", "rf_ele", "deg", None),
    ("ln preferred depth", "preferred_depth_closedloop", "ln units", np.log),
]:
    values = fov_cells[col].to_numpy(float)
    g = roi_location.spatial_gradient(
        fov_cells["center_x"],
        fov_cells["center_y"],
        transform(values) if transform else values,
    )
    print(
        f"{label:>18}: n = {g['n']}, R2 = {g['r2']:.3f}, F-test p = {g['pval_f']:.2e}, "
        f"shuffle p = {g['pval_perm']:.1e}, "
        f"gradient = {g['magnitude'] / um_per_px * 100:.3f} {unit}/100 um "
        f"at {g['direction']:.1f} deg"
    )

In [ ]:
import scipy

x = fov_cells["center_x"]
y = fov_cells["center_y"]
values = fov_cells["rf_azi"]
seed = 42
n_perm = 10000


x, y, values = (np.asarray(a, dtype=float) for a in (x, y, values))
ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(values)
X = np.column_stack([np.ones(ok.sum()), x[ok], y[ok]])
v = values[ok]
n = len(v)
rng = np.random.default_rng(seed)
# Column 0 is the data, the others its permutations, so one lstsq fits them all
V = np.column_stack([v] + [rng.permutation(v) for _ in range(n_perm)])
coefs = np.linalg.lstsq(X, V, rcond=None)[0]
resid = V - X @ coefs
r2 = 1 - np.sum(resid**2, axis=0) / np.sum((v - v.mean()) ** 2)
f_stat = (r2[0] / 2) / ((1 - r2[0]) / (n - 3))
slope_x, slope_y = coefs[1:, 0]
out = dict(
    n=n,
    slope_x=slope_x,
    slope_y=slope_y,
    magnitude=np.hypot(slope_x, slope_y),
    direction=np.degrees(np.arctan2(slope_y, slope_x)),
    r2=r2[0],
    pval_f=scipy.stats.f.sf(f_stat, 2, n - 3),
    pval_perm=(np.sum(r2[1:] >= r2[0]) + 1) / (n_perm + 1),
)

## Panels I, J, K: RF Similarity vs Cortical Distance

In [ ]:
YLABELS = [
    r"Mean |$\Delta$(RF azimuth)| (degrees)",
    r"Mean |$\Delta$(RF elevation)| (degrees)",
    r"Mean |$\Delta\log_2$(preferred virtual depth)|",
]

In [ ]:
fig = plt.figure(figsize=(18 / 2.54, 4.5 / 2.54), dpi=300)
axes = rf_distance.plot_rf_distance_panels(
    fig,
    [[0.08 + i * 0.33, 0.2, 0.24, 0.7] for i in range(3)],
    rf_dist_centers,
    rf_dist_means,
    rf_dist_ci_low,
    rf_dist_ci_high,
    FONTSIZE_DICT,
    ylabels=YLABELS,
    null=rf_dist_null,
)
axes[-1].set_ylim(0, 4)
axes[-1].set_yticks([0, 4])
style.savefig(
    SAVE_ROOT / f"fig_receptive_fields_rf_distance{OUT_SFX}.svg", bbox_inches="tight", fig=fig
)
plt.show()

In [ ]:
# Export statistics for manuscript tracking (panels E and I, J, K).
# Every value is read from the data the panels plot: depth_neurons for E, the
# pairwise data for I-K.
from v1_depth_map.stats import export_figure_stats

rf_dist_n_neurons = rf_distance.count_neurons(pairwise_df).sum()
pop_sessions = depth_neurons["session"].unique()
pop_n_mice = pd.Series(pop_sessions).str.split("_").str[0].nunique()
stats_fig4 = {
    "pop_n_neurons": {
        "raw": len(depth_neurons),
        "formatted": f"{len(depth_neurons):,}",
        "latex_macro": "statFigFourPopNeurons",
        "description": "Depth-tuned neurons with significant multi-depth RFs (Fig 4E)",
    },
    "pop_n_sessions": {
        "raw": len(pop_sessions),
        "formatted": f"{len(pop_sessions)}",
        "latex_macro": "statFigFourPopSessions",
        "description": "Multi-depth sessions contributing neurons (Fig 4E)",
    },
    "pop_n_mice": {
        "raw": int(pop_n_mice),
        "formatted": f"{pop_n_mice}",
        "latex_macro": "statFigFourPopMice",
        "description": "Mice contributing neurons (Fig 4E)",
    },
    "rf_dist_n_neurons": {
        "raw": int(rf_dist_n_neurons),
        "formatted": f"{rf_dist_n_neurons:,}",
        "latex_macro": "statFigFourRFDistNeurons",
        "description": "Depth-tuned neurons with significant RFs in pairwise distance analysis (Fig 4I-K)",
    },
    "rf_dist_n_sessions": {
        "raw": rf_dist_counts["n_sessions"],
        "formatted": f"{rf_dist_counts['n_sessions']}",
        "latex_macro": "statFigFourRFDistSessions",
        "description": "Sessions in pairwise distance analysis (Fig 4I-K)",
    },
    "rf_dist_n_mice": {
        "raw": rf_dist_counts["n_mice"],
        "formatted": f"{rf_dist_counts['n_mice']}",
        "latex_macro": "statFigFourRFDistMice",
        "description": "Mice in pairwise distance analysis (Fig 4I-K)",
    },
    "rf_dist_n_pairs": {
        "raw": rf_dist_counts["n_pairs"],
        "formatted": f"{rf_dist_counts['n_pairs']:,}",
        "latex_macro": "statFigFourRFDistPairs",
        "description": "Neuron pairs more than 10 um apart, within the binned range (Fig 4I-K)",
    },
}
if OUT_SFX == "_spks":  # the manuscript uses the RFs fitted on deconvolved spikes
    export_figure_stats("figure4_receptive_fields", stats_fig4, figure="fig4")
else:
    print("RF variant: manuscript stats not exported")


for k, v in stats_fig4.items():
    print(f"{k}: {v['raw']} ({v['formatted']}) - {v['description']}")

## Full Figure 4: Unified Publication Layout (Panels A–K)

This cell compiles all individual panels into a single, cohesive publication figure canvas (18.5 × 21.5 cm):
- **A**: Model formulation schematic
- **B**: Multi-depth reconstructed visual stimulus frames across 8 depths
- **C**: Depth tuning curves & 8-depth RF heatmaps for 3 example V1 neurons
- **D**: 3D receptive fields as nested isosurfaces (50-98% of peak) + faint floor/wall marginal projections
- **E**: Population correlation (RF preferred depth vs trial-averaged preferred depth)
- **F, G, H**: Spatial retinotopy in example V1 FOV (Azimuth, Elevation, Preferred Depth + meanImg inset)
- **I, J, K**: Mean pairwise RF azimuth / elevation / log₂ preferred depth distance vs distance between cells

In [ ]:
# Assembled figure, 18.5 x 21.5 cm. All positions are in cm from the bottom-left
# corner of the canvas; `rect_cm` converts them to figure fractions. Panel A is
# left blank for the model schematic, which is dropped in afterwards.
# Panels A-H were laid out on a 17.0 cm canvas; `_up` lifts them by ROW_H to
# make room for the I-K strip at the bottom.
ROW_H = 4.5
FIG_W, FIG_H = 18.5, 17.0 + ROW_H


def _up(rect):
    """Shift an (x, y, ...) position up by ROW_H."""
    return (rect[0], rect[1] + ROW_H, *rect[2:])


LAYOUT = dict(
    letters={
        **{
            k: _up(v)
            for k, v in {
                "A": (0.15, 16.6),
                "B": (0.15, 14.4),
                "C": (2.10, 16.6),
                "D": (7.60, 16.6),
                "E": (11.00, 10.6),
                "F": (0.15, 5.75),
                "G": (6.10, 5.75),
                "H": (12.05, 5.75),
            }.items()
        },
        "I": (0.15, 4.25),
        "J": (6.10, 4.25),
        "K": (12.05, 4.25),
    },
    stimulus=_up((0.75, 13.6, 1.25, 7.2)),  # x, top row bottom, width, total height
    tuning=_up((2.75, 15.8, 1.20, 0.95)),
    rf_maps=_up((2.70, 13.6, 1.25, 7.2)),
    example_dx=1.55,
    volume=_up((7.5, 10.6, 10.4, 6)),
    population=_up((12.0, 6.3, 5.2, 4.0)),
    fov=[
        _up(r)
        for r in [
            (0.35, 0.35, 5.75, 5.2),
            (6.30, 0.35, 5.75, 5.2),
            (12.25, 0.35, 5.75, 5.2),
        ]
    ],
    fov_inset=_up((15.45, 3.65, 1.75, 1.75)),
    rf_distance=[
        (1.30, 0.90, 4.40, 3.10),
        (7.25, 0.90, 4.40, 3.10),
        (13.20, 0.90, 4.40, 3.10),
    ],
)

depths = sorted(neurons_df.best_depth.unique())
depths_arr = np.array(depths)
ndepths = len(depths)

frames_contra = frames_rf[..., frames_rf.shape[-1] // 2 :]
has_data = np.any(frames_contra, axis=(2, 3))
start_frame = np.where(np.sum(has_data, axis=0) == ndepths)[0][2450]

fig = plt.figure(figsize=(FIG_W / 2.54, FIG_H / 2.54), dpi=300)
ax = fig.add_axes([0, 0, 1, 1])
ax.set_xticks([])
ax.set_yticks([])
fig.patch.set_facecolor("white")
for letter, (x, y) in LAYOUT["letters"].items():
    panel_letter(fig, letter, x, y)

# B: multi-depth stimulus frames
plot_stimulus_frames(
    fig, rect_cm(fig, *LAYOUT["stimulus"]), frames_contra, start_frame, depths
)

# C: example neuron tuning curves and RF maps
axes = plot_example_neurons(
    fig,
    neurons_df=neurons_df_example,
    trials_df=trials_df_example,
    tuning_rect=rect_cm(fig, *LAYOUT["tuning"]),
    rf_rect=rect_cm(fig, *LAYOUT["rf_maps"]),
    dx=LAYOUT["example_dx"] / FIG_W,
    ndepths=ndepths,
)

# E: Population correlation (scatter plot below 3D volume)
plot_rf_vs_tuning_depth(fig, depth_neurons, rect_cm(fig, *LAYOUT["population"]))

# D: 3D receptive-field volumes (to the right of example neurons)
plot_rf_volumes_3d(
    fig.add_axes(rect_cm(fig, *LAYOUT["volume"]), projection="3d"),
    neurons_df_example,
    depths_arr,
    tickpad=-4,
    labelpad=-8,
    elev=40,
    azim=-35,
)

# F, G, H: retinotopy across the example FOV
plot_fov_retinotopy(
    fig,
    rects=[rect_cm(fig, *r) for r in LAYOUT["fov"]],
    neurons_df_fov=neurons_df_fov,
    stat_fov=stat_fov,
    ops_fov=ops_fov,
    depths=depths,
    inset_rect=rect_cm(fig, *LAYOUT["fov_inset"]),
)

# I, J, K: pairwise RF / depth distance vs distance between cells
axes = rf_distance.plot_rf_distance_panels(
    fig,
    [rect_cm(fig, *r) for r in LAYOUT["rf_distance"]],
    rf_dist_centers,
    rf_dist_means,
    rf_dist_ci_low,
    rf_dist_ci_high,
    FONTSIZE_DICT,
    ylabels=YLABELS,
    null=rf_dist_null,
)
axes[-1].set_ylim(0, 4)
axes[-1].set_yticks([0, 4])

for suffix in ("svg", "pdf"):
    style.savefig(
        SAVE_ROOT / f"fig_receptive_fields_full{OUT_SFX}.{suffix}",
        fig=fig,
        dpi=600,  # resolution of the rasterised 3D volumes
    )
plt.show()